# Week 2-2: 보너스 Transform - 문자열 리스트를 관계 테이블로 바꾸기

이 노트북은 `w2_1` 이후에 진행하는 보너스/심화 실습입니다.

`w2_1`이 저장 전 품질 점검과 clean table 만들기였다면, `w2_2`는 원본 안에 뭉쳐 있는 복잡한 값을 **분석 가능한 관계 테이블**로 바꾸는 Transform입니다.

핵심은 pandas 문법을 많이 외우는 것이 아니라, 다음 구조 변화를 이해하는 것입니다.

`영화 1행 안에 여러 값이 들어 있는 구조` → `관계 1개당 1행인 테이블`


## 실습 목표

- `credits.csv`, `keywords.csv`가 어떤 데이터인지 눈으로 확인한다
- 영화 ID 기준으로 서로 다른 원천 데이터를 같은 분석 단위로 맞춘다
- 문자열처럼 저장된 리스트 컬럼을 Python 리스트로 파싱한다
- `keywords`, `cast`, `crew`를 각각 관계 테이블로 펼친다
- 키워드-영화, 배우-키워드, 감독-키워드 관계를 확인한다
- 나중에 DB에 Load한다면 어떤 테이블로 저장할지 생각해본다


## 1. 준비

In [1]:
import ast
from pathlib import Path

import pandas as pd


In [2]:
DATA_DIR_CANDIDATES = [Path("../data"), Path("week2/data"), Path("data")]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. download_dataset.py를 먼저 실행하세요.")

RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = RAW_DIR / "movies_metadata.csv"
MOVIES_CLEAN_PATH = OUTPUT_DIR / "movies_clean_for_load.csv"
CREDITS_PATH = RAW_DIR / "credits.csv"
KEYWORDS_PATH = RAW_DIR / "keywords.csv"

DATA_DIR


PosixPath('../data')

## 2. w2_1 결과물 읽기

이번 보너스 실습은 `w2_1`에서 만든 `movies_clean_for_load.csv`를 기준 영화 테이블처럼 사용합니다.

`movies_clean`은 영화 1개당 1행인 신뢰성 있는 정제 테이블이고, 이후 만드는 `movie_keywords`, `movie_cast`, `movie_crew`는 이 테이블의 `movie_id`와 연결됩니다.


In [3]:
if MOVIES_CLEAN_PATH.exists():
    movies_clean = pd.read_csv(MOVIES_CLEAN_PATH, parse_dates=["release_date"])
else:
    movies_raw = pd.read_csv(METADATA_PATH, low_memory=False)
    movies_clean = movies_raw[["id", "title", "release_date", "vote_average", "vote_count"]].copy()
    movies_clean = movies_clean.rename(columns={"id": "movie_id"})
    movies_clean["movie_id"] = pd.to_numeric(movies_clean["movie_id"], errors="coerce")
    movies_clean["release_date"] = pd.to_datetime(movies_clean["release_date"], errors="coerce")
    movies_clean["vote_average"] = pd.to_numeric(movies_clean["vote_average"], errors="coerce")
    movies_clean["vote_count"] = pd.to_numeric(movies_clean["vote_count"], errors="coerce")
    movies_clean = movies_clean.dropna(subset=["movie_id", "title"]).drop_duplicates("movie_id")
    movies_clean["movie_id"] = movies_clean["movie_id"].astype("int64")
    movies_clean["release_year"] = movies_clean["release_date"].dt.year

print("movies_clean shape:", movies_clean.shape)
movies_clean.head(3)


movies_clean shape: (43565, 11)


,movie_id,title,release_date,release_year,original_language,runtime,budget,revenue,popularity,vote_average,vote_count
0,862,Toy Story,1995-10-30,1995,en,81.0,30000000.0,373554033.0,21.946943,7.7,5415.0
1,8844,Jumanji,1995-12-15,1995,en,104.0,65000000.0,262797249.0,17.015539,6.9,2413.0
2,15602,Grumpier Old Men,1995-12-22,1995,en,101.0,0.0,0.0,11.712900,6.5,92.0


## 3. credits / keywords 원본 확인

`credits.csv`와 `keywords.csv`는 둘 다 영화별 부가 정보를 담고 있습니다.

- `credits`: 출연진(`cast`)과 제작진(`crew`)
- `keywords`: 영화에 붙은 키워드 목록

두 파일의 공통점은 한 영화 안에 여러 값이 리스트처럼 묶여 있다는 점입니다. 이 구조는 사람이 보기에는 괜찮지만, DB에서 집계하거나 조인하기에는 불편합니다.


In [4]:
credits_raw = pd.read_csv(CREDITS_PATH)
keywords_raw = pd.read_csv(KEYWORDS_PATH)

print("credits_raw shape:", credits_raw.shape)
print("keywords_raw shape:", keywords_raw.shape)


credits_raw shape: (45476, 3)
keywords_raw shape: (46419, 2)


In [5]:
display(credits_raw.head(2))
display(keywords_raw.head(2))


,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844


,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."


## 4. 최소 표준화: 영화 ID 이름 맞추기

`credits.csv`와 `keywords.csv`의 `id`는 영화 ID입니다. `movies_clean`에서는 같은 의미를 `movie_id`라고 부르므로 이름을 맞춥니다.

이 단계의 목적은 예쁜 컬럼명을 만드는 것이 아니라, 이후 조인에서 같은 의미의 컬럼을 분명히 하는 것입니다.


In [6]:
credits = credits_raw.rename(columns={"id": "movie_id"}).copy()
keywords = keywords_raw.rename(columns={"id": "movie_id"}).copy()

credits["movie_id"] = pd.to_numeric(credits["movie_id"], errors="coerce")
keywords["movie_id"] = pd.to_numeric(keywords["movie_id"], errors="coerce")

credits = credits.dropna(subset=["movie_id"]).copy()
keywords = keywords.dropna(subset=["movie_id"]).copy()
credits["movie_id"] = credits["movie_id"].astype("int64")
keywords["movie_id"] = keywords["movie_id"].astype("int64")

display(credits.head(2))
display(keywords.head(2))


,cast,crew,movie_id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844


,movie_id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."


## 5. 중복 확인 후 영화 단위로 맞추기

이번 실습에서는 영화 1개당 `credits` 1행, `keywords` 1행이 되도록 맞춘 뒤 조인합니다.

중복이 있다고 바로 에러로 보지는 않지만, 조인 전에 중복 여부를 확인하고 어떤 기준으로 정리할지 정해야 합니다. 여기서는 기본 실습처럼 첫 번째 행만 남깁니다.


In [7]:
duplicate_summary = pd.DataFrame({
    "table": ["credits", "keywords"],
    "duplicate_movie_id_rows": [
        credits["movie_id"].duplicated(keep=False).sum(),
        keywords["movie_id"].duplicated(keep=False).sum(),
    ],
})

duplicate_summary


,table,duplicate_movie_id_rows
0,credits,87
1,keywords,1972


In [8]:
credits_one = credits.drop_duplicates(subset=["movie_id"], keep="first")
keywords_one = keywords.drop_duplicates(subset=["movie_id"], keep="first")

credits_keywords = credits_one.merge(
    keywords_one,
    on="movie_id",
    how="inner",
    validate="one_to_one",
)

print("credits_keywords shape:", credits_keywords.shape)
credits_keywords[["movie_id", "keywords"]].head(3)


credits_keywords shape: (45432, 4)


,movie_id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."


## 6. 문자열 리스트 파싱 helper

`cast`, `crew`, `keywords`는 리스트처럼 보이지만 CSV 안에서는 문자열입니다.

예를 들어 `keywords`의 한 칸에는 이런 값이 들어 있습니다.

`[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}]`

이 문자열을 실제 Python 리스트로 바꿔야 각 키워드의 `id`, `name`을 꺼낼 수 있습니다.


In [9]:
def parse_list_column(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []
    return parsed if isinstance(parsed, list) else []

sample_keywords = parse_list_column(keywords_one.iloc[0]["keywords"])
sample_keywords[:3]


[{'id': 931, 'name': 'jealousy'},
 {'id': 4290, 'name': 'toy'},
 {'id': 5202, 'name': 'boy'}]

## 7. movie_keywords 만들기

원본 `keywords`는 영화 1행 안에 여러 키워드가 묶인 구조입니다. 이것을 `영화-키워드 조합 1개당 1행`으로 바꿉니다.

변환 전:

`movie_id = 862, keywords = [jealousy, toy, boy, ...]`

변환 후:

`movie_id = 862, keyword_name = jealousy`

`movie_id = 862, keyword_name = toy`

이렇게 바꾸면 키워드별 영화 수, 키워드별 평점, 특정 키워드 영화 목록을 쉽게 구할 수 있습니다.


In [10]:
def make_movie_keywords(keywords_df):
    expanded = keywords_df.copy()
    expanded["keywords"] = expanded["keywords"].map(parse_list_column)
    expanded = expanded.explode("keywords").dropna(subset=["keywords"])

    result = pd.DataFrame({
        "movie_id": expanded["movie_id"],
        "keyword_id": expanded["keywords"].map(lambda item: item.get("id") if isinstance(item, dict) else None),
        "keyword_name": expanded["keywords"].map(lambda item: item.get("name") if isinstance(item, dict) else None),
    })

    result = result.dropna(subset=["movie_id", "keyword_id", "keyword_name"]).copy()
    result["movie_id"] = result["movie_id"].astype("int64")
    result["keyword_id"] = result["keyword_id"].astype("int64")
    return result

movie_keywords = make_movie_keywords(keywords_one)

print("movie_keywords shape:", movie_keywords.shape)
movie_keywords.head(10)


movie_keywords shape: (156602, 3)


,movie_id,keyword_id,keyword_name
0,862,931,jealousy
0,862,4290,toy
0,862,5202,boy
0,862,6054,friendship
0,862,9713,friends
0,862,9823,rivalry
0,862,165503,boy next door
0,862,170722,new toy
0,862,187065,toy comes to life
1,8844,10090,board game


## 8. 자주 등장하는 키워드

`movie_keywords`는 키워드 하나가 한 행이므로 `groupby`로 키워드별 영화 수를 바로 셀 수 있습니다.


In [11]:
top_keywords = (
    movie_keywords
    .groupby("keyword_name", as_index=False)
    .agg(movie_count=("movie_id", "nunique"))
    .sort_values("movie_count", ascending=False)
)

top_keywords.head(20)


,keyword_name,movie_count
19592,woman director,3039
8823,independent film,1914
11651,murder,1285
1353,based on novel,822
11716,musical,726
15806,sex,679
19018,violence,647
12297,nudity,629
14797,revenge,618
1682,biography,613


## 9. movie_cast 만들기

`cast`도 영화 1행 안에 여러 배우가 들어 있는 구조입니다. 이것을 `영화-배우 조합 1개당 1행`으로 바꿉니다.

결과 테이블의 grain은 다음입니다.

`movie_id + actor_id + cast_order`


In [12]:
def make_movie_cast(credits_df):
    expanded = credits_df[["movie_id", "cast"]].copy()
    expanded["cast"] = expanded["cast"].map(parse_list_column)
    expanded = expanded.explode("cast").dropna(subset=["cast"])

    result = pd.DataFrame({
        "movie_id": expanded["movie_id"],
        "actor_id": expanded["cast"].map(lambda item: item.get("id") if isinstance(item, dict) else None),
        "actor_name": expanded["cast"].map(lambda item: item.get("name") if isinstance(item, dict) else None),
        "character": expanded["cast"].map(lambda item: item.get("character") if isinstance(item, dict) else None),
        "cast_order": expanded["cast"].map(lambda item: item.get("order") if isinstance(item, dict) else None),
    })

    result = result.dropna(subset=["movie_id", "actor_id", "actor_name"]).copy()
    result["movie_id"] = result["movie_id"].astype("int64")
    result["actor_id"] = result["actor_id"].astype("int64")
    return result

movie_cast = make_movie_cast(credits_one)

print("movie_cast shape:", movie_cast.shape)
movie_cast.head(10)


movie_cast shape: (562044, 5)


,movie_id,actor_id,actor_name,character,cast_order
0,862,31,Tom Hanks,Woody (voice),0
0,862,12898,Tim Allen,Buzz Lightyear (voice),1
0,862,7167,Don Rickles,Mr. Potato Head (voice),2
0,862,12899,Jim Varney,Slinky Dog (voice),3
0,862,12900,Wallace Shawn,Rex (voice),4
0,862,7907,John Ratzenberger,Hamm (voice),5
0,862,8873,Annie Potts,Bo Peep (voice),6
0,862,1116442,John Morris,Andy (voice),7
0,862,12901,Erik von Detten,Sid (voice),8
0,862,12133,Laurie Metcalf,Mrs. Davis (voice),9


## 10. movie_crew 만들기

`crew`는 제작진 정보입니다. 감독, 작가, 프로듀서, 음악, 편집 등 여러 역할이 들어 있습니다.

여기서는 전체 제작진 테이블인 `movie_crew`를 만들고, 그중 `job == "Director"`인 행만 따로 `directors`로 뽑습니다.


In [13]:
def make_movie_crew(credits_df):
    expanded = credits_df[["movie_id", "crew"]].copy()
    expanded["crew"] = expanded["crew"].map(parse_list_column)
    expanded = expanded.explode("crew").dropna(subset=["crew"])

    result = pd.DataFrame({
        "movie_id": expanded["movie_id"],
        "crew_id": expanded["crew"].map(lambda item: item.get("id") if isinstance(item, dict) else None),
        "crew_name": expanded["crew"].map(lambda item: item.get("name") if isinstance(item, dict) else None),
        "department": expanded["crew"].map(lambda item: item.get("department") if isinstance(item, dict) else None),
        "job": expanded["crew"].map(lambda item: item.get("job") if isinstance(item, dict) else None),
    })

    result = result.dropna(subset=["movie_id", "crew_id", "crew_name", "job"]).copy()
    result["movie_id"] = result["movie_id"].astype("int64")
    result["crew_id"] = result["crew_id"].astype("int64")
    return result

movie_crew = make_movie_crew(credits_one)
directors = movie_crew[movie_crew["job"] == "Director"].copy()

print("movie_crew shape:", movie_crew.shape)
print("directors shape:", directors.shape)
movie_crew.head(10)


movie_crew shape: (463836, 5)
directors shape: (48999, 5)


,movie_id,crew_id,crew_name,department,job
0,862,7879,John Lasseter,Directing,Director
0,862,12891,Joss Whedon,Writing,Screenplay
0,862,7,Andrew Stanton,Writing,Screenplay
0,862,12892,Joel Cohen,Writing,Screenplay
0,862,12893,Alec Sokolow,Writing,Screenplay
0,862,12894,Bonnie Arnold,Production,Producer
0,862,12895,Ed Catmull,Production,Executive Producer
0,862,12896,Ralph Guggenheim,Production,Producer
0,862,12897,Steve Jobs,Production,Executive Producer
0,862,8,Lee Unkrich,Editing,Editor


## 11. 키워드와 메타데이터 연결

이제 `movie_keywords`와 `movies_clean`을 `movie_id`로 연결합니다. 그러면 키워드만 있는 테이블에 영화 제목, 개봉연도, 평점 같은 메타데이터를 붙일 수 있습니다.


In [14]:
keyword_movies = movie_keywords.merge(
    movies_clean,
    on="movie_id",
    how="inner",
)

keyword_movies[["keyword_name", "title", "release_year", "vote_average", "vote_count"]].head(20)


,keyword_name,title,release_year,vote_average,vote_count
0,jealousy,Toy Story,1995,7.7,5415.0
1,toy,Toy Story,1995,7.7,5415.0
2,boy,Toy Story,1995,7.7,5415.0
3,friendship,Toy Story,1995,7.7,5415.0
4,friends,Toy Story,1995,7.7,5415.0
5,rivalry,Toy Story,1995,7.7,5415.0
6,boy next door,Toy Story,1995,7.7,5415.0
7,new toy,Toy Story,1995,7.7,5415.0
8,toy comes to life,Toy Story,1995,7.7,5415.0
9,board game,Jumanji,1995,6.9,2413.0


## 12. 키워드별 평균 평점 보기

키워드별로 어떤 영화들이 있고 평균 평점이 어떤지 확인합니다.

단, 영화 수가 너무 적은 키워드는 평균이 불안정하므로 여기서는 `movie_count >= 20`인 키워드만 봅니다.


In [15]:
keyword_rating_summary = (
    keyword_movies
    .dropna(subset=["vote_average"])
    .groupby("keyword_name", as_index=False)
    .agg(
        movie_count=("movie_id", "nunique"),
        avg_vote_average=("vote_average", "mean"),
        total_vote_count=("vote_count", "sum"),
    )
    .query("movie_count >= 20")
    .sort_values(["avg_vote_average", "movie_count"], ascending=[False, False])
)

keyword_rating_summary.head(20)


,keyword_name,movie_count,avg_vote_average,total_vote_count
14176,rage and hate,21,7.161905,28919.0
18664,unsociability,36,7.130556,17537.0
8820,individual,39,7.107692,27573.0
3335,class,20,7.090000,11863.0
14800,right and justice,21,7.052381,10998.0
6938,french noir,29,7.051724,1564.0
19087,war crimes,36,7.047222,15274.0
3693,concentration camp,20,7.030000,11857.0
13227,pixar animation,23,7.000000,3962.0
10727,marvel cinematic universe,20,6.980000,101479.0


## 13. 배우와 키워드 관계 보기

`movie_cast`와 `movie_keywords`를 연결하면 배우가 어떤 키워드의 영화에 자주 등장했는지 볼 수 있습니다.

예를 들어 특정 배우가 `martial arts`, `revenge`, `musical` 같은 키워드와 자주 연결되는지 확인할 수 있습니다.


In [16]:
actor_keyword = (
    movie_cast[["movie_id", "actor_name"]]
    .merge(movie_keywords[["movie_id", "keyword_name"]], on="movie_id", how="inner")
)

actor_keyword_summary = (
    actor_keyword
    .groupby(["actor_name", "keyword_name"], as_index=False)
    .agg(movie_count=("movie_id", "nunique"))
    .query("movie_count >= 3")
    .sort_values("movie_count", ascending=False)
)

actor_keyword_summary.head(20)


,actor_name,keyword_name,movie_count
1031974,Jackie Chan,martial arts,39
233356,Bess Flowers,musical,38
2281347,Stan Lee,marvel comic,30
981272,Huntz Hall,bowery boys,28
1478440,Leo Gorcey,bowery boys,27
796843,Fred Astaire,musical,27
2281432,Stan Lee,superhero,27
850309,Georges Méliès,silent film,26
2182488,Sammo Hung,martial arts,25
2247529,Shintarô Katsu,samurai,24


## 14. 감독과 키워드 관계 보기

`directors`와 `movie_keywords`를 연결하면 감독이 자주 다루는 키워드도 볼 수 있습니다.

이 분석은 완벽한 영화 해석이라기보다, 관계 테이블로 바꿨을 때 어떤 질문을 SQL/집계로 던질 수 있는지 보여주는 예시입니다.


In [17]:
director_keyword = (
    directors[["movie_id", "crew_name"]]
    .rename(columns={"crew_name": "director_name"})
    .merge(movie_keywords[["movie_id", "keyword_name"]], on="movie_id", how="inner")
)

director_keyword_summary = (
    director_keyword
    .groupby(["director_name", "keyword_name"], as_index=False)
    .agg(movie_count=("movie_id", "nunique"))
    .query("movie_count >= 2")
    .sort_values("movie_count", ascending=False)
)

director_keyword_summary.head(20)


,director_name,keyword_name,movie_count
49587,Georges Méliès,silent film,33
113723,Penelope Spheeris,woman director,18
5368,Alfred Hitchcock,suspense,18
131505,Ryan Polito,stand-up comedy,17
58652,Ishirô Honda,kaiju,17
95894,Martha Coolidge,woman director,16
154465,William Beaudine,bowery boys,16
156642,Woody Allen,independent film,15
64180,Jay Karas,stand-up comedy,15
104728,Mira Nair,woman director,15


## 15. Load로 이어지는 테이블 설계

보너스 실습의 결과는 보고용 표라기보다 DB에 저장 가능한 관계 테이블 후보입니다.

Load 전에 각 테이블의 grain과 key를 정리해두면, 어떤 중복을 허용하고 어떤 중복을 막아야 하는지 판단하기 쉬워집니다.


In [18]:
load_plan = pd.DataFrame([
    {
        "table_name": "movie_keywords",
        "dataframe": "movie_keywords",
        "grain": "영화-키워드 조합 1개당 1행",
        "key": "movie_id + keyword_id",
    },
    {
        "table_name": "movie_cast",
        "dataframe": "movie_cast",
        "grain": "영화-배우 조합 1개당 1행",
        "key": "movie_id + actor_id + cast_order",
    },
    {
        "table_name": "movie_crew",
        "dataframe": "movie_crew",
        "grain": "영화-제작진 역할 1개당 1행",
        "key": "movie_id + crew_id + job",
    },
])

load_plan


,table_name,dataframe,grain,key
0,movie_keywords,movie_keywords,영화-키워드 조합 1개당 1행,movie_id + keyword_id
1,movie_cast,movie_cast,영화-배우 조합 1개당 1행,movie_id + actor_id + cast_order
2,movie_crew,movie_crew,영화-제작진 역할 1개당 1행,movie_id + crew_id + job


## 16. 정리

이번 보너스 실습에서는 raw 문자열 컬럼을 그대로 두지 않고 관계 테이블로 바꿨습니다.

- `keywords` → `movie_keywords`
- `cast` → `movie_cast`
- `crew` → `movie_crew`

핵심은 `explode`라는 함수 이름이 아니라, **한 행 안에 뭉쳐 있던 여러 값을 분석 가능한 여러 행으로 바꾼다**는 Transform 감각입니다.
